# Customer Churn Intelligence Platform

##Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

##Load Dataset

In [ ]:
df=pd.read_csv('Telco_customer_churn.csv')

In [ ]:
df.head()

##Dataset Shape and Information

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.info()

In [ ]:
df['Total Charges'].dtype

In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [ ]:
df.columns.tolist()

##Missing values

In [ ]:
df.isnull().sum().sort_values(ascending=False)

##Duplicate Records

In [ ]:
df.duplicated().sum()

##Remove Data Leakage Features

In [ ]:
drop_cols=['CustomerID','Count','Churn Value','Churn Score','CLTV','Churn Reason']
df=df.drop(columns=drop_cols)

In [ ]:
df.shape

## Missing Value Check

In [ ]:
df.isnull().sum().sort_values(ascending=False)

No missing values were found after removing the Churn Reason column.

#Exploratory Data Analysis(EDA)

## Target Variable Distribution

In [ ]:
df['Churn Label'].value_counts()

In [ ]:
(df['Churn Label'].value_counts(normalize=True)*100)

## Churn Distribution Plot

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    x='Churn Label',
    data=df
)

plt.show()

Observation  
The plot compares the number of churned and non-churned customers.
Non-churned customers are likely more numerous than churned customers.

Business Insight  
Most customers remain with the company, while a smaller group churns.
Understanding the characteristics of churned customers can help the company improve retention strategies and reduce customer loss.

## Numerical Features Summary

In [ ]:
num_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges']
df[num_cols].describe()

## Tenure Distribution

In [ ]:
sns.histplot(df['Tenure Months'], kde=True)
plt.xlabel("Tenure Months")
plt.ylabel("Number of Customers")
plt.title("Distribution of Customer Tenure")
plt.show()

Observation:

The tenure distribution is U-shaped, with a high concentration of customers in the early months (0–5) and the later months (65–72).
Relatively fewer customers fall within the middle tenure range.

Business Insight:

The customer base consists of both newly acquired and long-term loyal customers.
New customers may be more susceptible to churn and require targeted onboarding and engagement strategies.
Long-tenure customers appear more stable, highlighting the importance of retaining customers during their initial months of service.

## Monthly Charges Distribution

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    df['Monthly Charges'],
    bins=20,
    kde=True
)
plt.show()


Observation:

Monthly Charges show a bimodal distribution, with customer concentrations in both low and high charge ranges.

Business Insight:

The presence of distinct pricing segments suggests that customer behavior and churn risk may vary across plans.

## Total Charges Distribution

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    df['Total Charges'],
    bins=20,
    kde=True
    )
plt.show()

Observation:  
Total Charges are right-skewed, with most customers having lower total charges and fewer customers having very high total charges.

Business Insight:  
Most customers have accumulated lower total charges, indicating a large proportion of relatively newer customers.
Customers with higher total charges are likely long-term customers and may represent higher lifetime value.

## Churn vs Tenure Months

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x='Churn Label',
    y='Tenure Months',
    data=df
)
plt.show()


Observation:

Customers who churned (Yes) generally have much lower tenure than customers who did not churn (No).
The median tenure of non-churned customers is significantly higher.

Business Insight:

Customers with longer tenure are more likely to stay with the company.
Customer retention efforts should focus on the early months of the customer lifecycle, where churn risk appears to be higher.

## Churn vs Monthly Charges

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x='Churn Label',
    y='Monthly Charges',
    data=df
)

plt.show()

Observation:

Customers who churned (Yes) tend to have higher monthly charges than customers who did not churn (No).
The median monthly charge is noticeably higher for churned customers.

Business Insight:

Higher monthly charges may be associated with a greater likelihood of churn.
The company may need to review pricing, service value, or retention strategies for higher-paying customers.

## Churn vs Total Charges

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x='Churn Label',
    y='Total Charges',
    data=df
)

plt.show()


Key Insight: High-Value Customer Vulnerability        

Analysis revealed that customer churn is heavily concentrated among high-spending tiers. The median total charges for churned accounts significantly exceed those of retained customers, indicating a critical revenue risk. This suggests a potential gap in pricing perception or ROI realization for top-tier clients, signaling an immediate need for targeted premium retention strategies.

## Contract vs Churn

In [ ]:
contract_churn = pd.crosstab(
    df['Contract'],
    df['Churn Label'],
    normalize='index'
)
contract_churn


In [ ]:
contract_churn.plot(
    kind='bar',
    stacked=True,
    figsize=(8,5)
)

plt.show()

Business Insight

Customers with longer contracts are significantly less likely to churn.
Month-to-month customers represent the highest churn risk and should be a primary focus of retention efforts.


## Internet Service vs Churn

In [ ]:
internet_churn = pd.crosstab(
    df['Internet Service'],
    df['Churn Label'],
    normalize='index'
)
internet_churn

In [ ]:
internet_churn.plot(
    kind='bar',
    stacked=True,
    figsize=(8,5)
)

plt.show()

Observation:

Fiber optic customers have the highest churn rate (41.9%), followed by DSL customers (19.0%).   
Customers without internet service have the lowest churn rate (7.4%).

Business Insight:

Fiber optic customers are significantly more likely to churn, indicating potential issues related to pricing, service quality, or customer expectations.   
The company should investigate and improve the experience of Fiber optic customers to reduce churn.

## Payment Method vs Churn

In [ ]:
payment_churn = pd.crosstab(
    df['Payment Method'],
    df['Churn Label'],
    normalize='index'
)

payment_churn

In [ ]:
payment_churn.plot(
    kind='bar',
    stacked=True,
    figsize=(10,5)
)

plt.show()

Observation:

Customers using Electronic Check have the highest churn rate (45.3%).
Customers using automatic payment methods have the lowest churn rates (15–17%).

Business Insight:

Customers who use Electronic Check are significantly more likely to churn.
Encouraging customers to switch to automatic payment methods may help improve retention and reduce churn.

In [ ]:
df['Target']= df['Churn Label'].map({
    'No':0,
    'Yes':1
})

## Correlation Matrix

In [ ]:
numeric_df= df.select_dtypes(
    include=['int64', 'float64']
)
plt.figure(figsize=(10,6))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm'
)
plt.show()

## Key Findings

- Customers with low tenure are more likely to churn.
- Month-to-month contracts show the highest churn.
- Higher monthly charges are associated with increased churn.
- Payment method and internet service type influence churn behavior.
- Tenure and billing-related features are important predictors.

# Feature Engineering

##Total Additional Services

In [ ]:
services_cols = ['Online Security',
                 'Online Backup',
                 'Device Protection',
                 'Tech Support',
                 'Streaming TV',
                 'Streaming Movies']
for col in services_cols:
    df[col] = np.where( df[col]== 'Yes',1,0
    )
df['Total Services'] = df[services_cols].sum(axis=1)

##Security Services

In [ ]:
security_cols = ['Online Security',
                 'Online Backup',
                 'Device Protection',
                 'Tech Support']
df['Security Services Count']= df[security_cols].sum(axis=1)

##Streaming Services

In [ ]:
stream_cols = [
    'Streaming TV',
    'Streaming Movies'
]

df['Streaming Services Count'] = df[stream_cols].sum(axis=1)

##Tenure Group

In [ ]:
df['Tenure Group'] = pd.cut(
    df['Tenure Months'],
    bins=[0,12,24,48,72],
    labels=[
        '0-12',
        '12-24',
        '24-48',
        '48-72'
    ]
)

##Average Revenue Per Month

In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)
df['Avg Revenue Per Month'] = (
    df['Total Charges']
    /
    (df['Tenure Months'] + 1)
)

Adding 1 to avoid 0 in the denominator

In [ ]:
df[['Total Services',
     'Security Services Count',
     'Streaming Services Count',
      'Tenure Group',
      'Avg Revenue Per Month'
     ]].head()

In [ ]:
df.columns.tolist()

In [ ]:
df['Online Security'].unique()

In [ ]:
df['Country'].nunique()

In [ ]:
df['State'].nunique()

In [ ]:
df['City'].nunique()

In [ ]:
df.drop(columns=['Country','State'],inplace=True)

##Check if feature engineering worked

In [ ]:
df[['Total Services',
    'Security Services Count',
    'Streaming Services Count',
    'Avg Revenue Per Month']].describe()

In [ ]:
df['Tenure Group'].value_counts()

In [ ]:
df.to_csv(
    "feature_engineered_data.csv",
    index=False
)

#Preprocessing

In [ ]:
X= df.drop('Target', axis=1)
X.dtypes


## Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## Load Feature Engineered Dataset

In [ ]:
df = pd.read_csv("feature_engineered_data.csv")

df.head()

## Remove Unnecessary Columns

In [ ]:
drop_cols = [
    'City',
    'Zip Code',
    'Lat Long',
    'Churn Label'
]

df = df.drop(columns=drop_cols)

## Verify Remaining Columns

In [ ]:
df.columns.tolist()

In [ ]:
df = df.drop(columns=['Latitude', 'Longitude'])

## Split Features and Target

In [ ]:
X = df.drop('Target', axis=1)

y = df['Target']

In [ ]:
print("X Shape:", X.shape)
print("y Shape:", y.shape)

## Identify Feature Types

In [ ]:
categorical_cols = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

In [ ]:
print("Categorical Columns")
print(categorical_cols)

print("\nNumerical Columns")
print(numerical_cols)

## Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
print(y_train.value_counts(normalize=True))

print("\n")

print(y_test.value_counts(normalize=True))

In [ ]:
X_train.shape
X_test.shape

In [ ]:
y_train.value_counts(normalize=True)

# Create Preprocessing Pipeline

##Numerical Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ('scaler', StandardScaler())
    ]
)

##Categorical Pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(
    steps=[
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore'
            )
        )
    ]
)

##Column Transformer

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            numeric_transformer,
            numerical_cols
        ),
        (
            'cat',
            categorical_transformer,
            categorical_cols
        )
    ]
)

##Fit and Transform

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

##Check Shapes

In [ ]:
print(X_train_processed.shape)

print(X_test_processed.shape)

# Logistic Regression Baseline Model

In [ ]:
from sklearn.linear_model import LogisticRegression

##Train Model

In [ ]:
lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr.fit(
    X_train_processed,
    y_train
)

##Predictions

In [ ]:
y_pred = lr.predict(X_test_processed)

y_prob = lr.predict_proba(
    X_test_processed
)[:,1]

##Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [ ]:
print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Precision:",
    precision_score(y_test, y_pred)
)

print(
    "Recall:",
    recall_score(y_test, y_pred)
)

print(
    "F1:",
    f1_score(y_test, y_pred)
)

print(
    "ROC AUC:",
    roc_auc_score(y_test, y_prob)
)

##Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

##Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.55      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



### Logistic Regression Results

The Logistic Regression model achieved an accuracy of 80.27% and a ROC-AUC score of 85.30%.

While the model performed well overall, it achieved a recall of 55.35% for churned customers, indicating that nearly half of the churners were missed.

This model serves as a strong baseline for comparison with more advanced models.

# Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(
    X_train_processed,
    y_train
)


DecisionTreeClassifier(random_state=42)

##Predictions

In [ ]:
y_pred_dt = dt.predict(X_test_processed)

y_prob_dt = dt.predict_proba(
    X_test_processed
)[:,1]

##Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:",
      accuracy_score(y_test, y_pred_dt))

print("Precision:",
      precision_score(y_test, y_pred_dt))

print("Recall:",
      recall_score(y_test, y_pred_dt))

print("F1:",
      f1_score(y_test, y_pred_dt))

print("ROC AUC:",
      roc_auc_score(y_test, y_prob_dt))

Accuracy: 0.730305180979418
Precision: 0.49242424242424243
Recall: 0.5213903743315508
F1: 0.5064935064935064
ROC AUC: 0.6631313131313131


##Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_dt
    )
)

              precision    recall  f1-score   support

           0       0.82      0.81      0.81      1035
           1       0.49      0.52      0.51       374

    accuracy                           0.73      1409
   macro avg       0.66      0.66      0.66      1409
weighted avg       0.74      0.73      0.73      1409



The Decision Tree model exhibited signs of overfitting, achieving significantly lower test performance than Logistic Regression.

The ROC-AUC score decreased from 0.853 to 0.663, indicating weaker discrimination between churners and non-churners.

# Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train_processed,
    y_train
)

##Predictions

In [ ]:
y_pred_rf = rf.predict(X_test_processed)

y_prob_rf = rf.predict_proba(
    X_test_processed
)[:,1]

##Evaluation

In [ ]:
print("Accuracy:",
      accuracy_score(y_test, y_pred_rf))

print("Precision:",
      precision_score(y_test, y_pred_rf))

print("Recall:",
      recall_score(y_test, y_pred_rf))

print("F1:",
      f1_score(y_test, y_pred_rf))

print("ROC AUC:",
      roc_auc_score(y_test, y_prob_rf))

##Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_rf
    )
)

## Model Comparison

Three machine learning models were evaluated for customer churn prediction.

| Model | Accuracy | Precision | Recall | F1 Score | ROC-AUC |
|---------|---------|---------|---------|---------|---------|
| Logistic Regression | 80.27% | 65.09% | 55.35% | 59.83% | 85.30% |
| Decision Tree | 73.03% | 49.24% | 52.14% | 50.65% | 66.31% |
| Random Forest | 78.85% | 62.42% | 51.07% | 56.18% | 83.21% |

### Key Findings

- Logistic Regression achieved the best overall performance.
- Decision Tree showed signs of overfitting and performed poorly on unseen data.
- Random Forest improved upon Decision Tree but did not outperform Logistic Regression.
- The ROC-AUC score of 85.30% indicates that Logistic Regression effectively distinguishes churning and non-churning customers.
- Recall remains relatively low, suggesting that many churning customers are still being missed.

# Handle Class Imbalance (SMOTE)


The dataset is imbalanced, with significantly fewer churned customers than non-churned customers.

To improve churn detection (Recall), SMOTE will be applied to balance the training data before retraining the models.

In [ ]:

pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

In [ ]:
print(y_train_smote.value_counts())

### Class Distribution After SMOTE

After applying SMOTE, both classes are now perfectly balanced.

- Class 0 (No churn): 4139 samples  
- Class 1 (Churn): 4139 samples  

This ensures the model does not get biased toward the majority class and improves fairness in prediction.

##Retrain Logistic Regression (After SMOTE)

Now we retrain Logistic Regression using the SMOTE-balanced training data.  
This helps the model learn both classes equally.

##Train Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_smote, y_train_smote)

##Predictions using Logistic Regression

In [ ]:
y_pred_log = log_model.predict(X_test_processed)

##Model Evaluation (Logistic Regression)
We check accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("Precision:", precision_score(y_test, y_pred_log))
print("Recall:", recall_score(y_test, y_pred_log))
print("F1 Score:", f1_score(y_test, y_pred_log))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_log))

##Decision Tree (After SMOTE)

Now we train a Decision Tree model on the SMOTE-balanced dataset and evaluate its performance.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_smote, y_train_smote)

##Predictions using Decision Tree

In [ ]:
y_pred_dt = dt_model.predict(X_test_processed)

##Model Evaluation (Decision Tree)
We evaluate accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("F1 Score:", f1_score(y_test, y_pred_dt))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))

##Random Forest (After SMOTE)

Now we train a Random Forest model on the SMOTE-balanced dataset and evaluate its performance.  
Random Forest usually performs better than a single Decision Tree because it reduces overfitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_smote, y_train_smote)

##Predictions using Random Forest

In [ ]:
y_pred_rf = rf_model.predict(X_test_processed)

##Model Evaluation (Random Forest)
We evaluate accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

##XGBoost Model (After SMOTE)

Now we train an XGBoost model on the SMOTE-balanced dataset.  
XGBoost is a powerful boosting algorithm that usually gives the best performance for classification problems like churn prediction.

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_model = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train_smote, y_train_smote)

##Predictions using XGBoost

In [ ]:
y_pred_xgb = xgb_model.predict(X_test_processed)

##Model Evaluation (XGBoost)
We evaluate accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))

##Model Comparison

Now we compare all models to select the best one based on performance metrics like Accuracy, Precision, Recall, and F1-score.

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ]
})

results

##Best Model Selection

From the comparison table, we select the model with the best F1-score (important for churn prediction).

In [ ]:
best_model = results.loc[results["F1 Score"].idxmax()]
best_model

##Feature Importance

Feature importance helps us understand which features are most responsible for predicting customer churn.  
Here we use the trained XGBoost model.

In [ ]:
import matplotlib.pyplot as plt

importances = xgb_model.feature_importances_
features = preprocessor.get_feature_names_out() # Corrected line

feat_df = pd.DataFrame({
    "Feature": features,
    "Importance": importances
})

feat_df = feat_df.sort_values(by="Importance", ascending=False)

feat_df

##Top Feature Importance Plot

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(feat_df["Feature"][:10], feat_df["Importance"][:10])
plt.gca().invert_yaxis()
plt.title("Top 10 Important Features (XGBoost)")
plt.show()

##Hyperparameter Tuning (XGBoost)

Now we tune XGBoost hyperparameters to improve model performance using RandomizedSearchCV.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42, eval_metric='logloss')

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=10,
    scoring="f1",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_smote, y_train_smote)

In [ ]:
best_xgb = search.best_estimator_
print(search.best_params_)

##Evaluation of Tuned Model

In [ ]:
y_pred_tuned = best_xgb.predict(X_test_processed)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall:", recall_score(y_test, y_pred_tuned))
print("F1 Score:", f1_score(y_test, y_pred_tuned))

##SHAP Explainability

SHAP (Shapley Additive explanations) helps us understand how each feature contributes to individual predictions as well as overall model behavior.

In [ ]:

import shap

In [ ]:
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test_processed)

##SHAP Summary Plot
This shows how each feature affects model predictions globally.

In [ ]:
shap.summary_plot(shap_values, X_test_processed)

## SHAP Insights

- Red points = high feature value
- Blue points = low feature value  
- Right side = increases churn probability  
- Left side = decreases churn probability  

##Individual Prediction Explanation
We explain why a single customer is predicted to churn or not.

In [ ]:
shap.initjs()

shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_test_processed[0] # Changed from X_test.iloc[0] to X_test_processed[0]
)

In [ ]:
X.columns.tolist()

In [ ]:
for col in [
    'Gender',
    'Senior Citizen',
    'Partner',
    'Dependents',
    'Phone Service',
    'Multiple Lines',
    'Internet Service',
    'Contract',
    'Paperless Billing',
    'Payment Method'
]:
    print("\n", col)
    print(df[col].unique())

# Model Deployment with Streamlit

## Save Model and Preprocessor

In [ ]:
import joblib

joblib.dump(best_xgb, "model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

print("Files saved successfully.")

## Verify Saved Files

In [ ]:
import os

os.listdir()

Create app.py

In [131]:
import streamlit as st
import pandas as pd
import joblib

# Load model and preprocessor
model = joblib.load("model.pkl")
preprocessor = joblib.load("preprocessor.pkl")

st.title("Customer Churn Intelligence Platform")

st.write("Predict whether a customer will churn or not.")

st.success("Model loaded successfully!")

2026-06-15 10:57:14.201 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 10:57:14.202 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 10:57:14.204 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 10:57:14.209 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 10:57:14.211 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 10:57:14.213 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 10:57:14.216 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-15 10:57:14.220 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [132]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# Load model and preprocessor
model = joblib.load("model.pkl")
preprocessor = joblib.load("preprocessor.pkl")

st.title("Customer Churn Intelligence Platform")

st.write("Predict whether a customer will churn or not.")

st.success("Model loaded successfully!")

st.header("Customer Information")

gender = st.selectbox("Gender", ["Male", "Female"])

senior_citizen = st.selectbox("Senior Citizen", ["No", "Yes"])

partner = st.selectbox("Partner", ["No", "Yes"])

dependents = st.selectbox("Dependents", ["No", "Yes"])

tenure_months = st.number_input(
    "Tenure Months",
    min_value=0,
    max_value=72,
    value=12
)

phone_service = st.selectbox(
    "Phone Service",
    ["Yes", "No"]
)

multiple_lines = st.selectbox(
    "Multiple Lines",
    ["No", "Yes", "No phone service"]
)

internet_service = st.selectbox(
    "Internet Service",
    ["DSL", "Fiber optic", "No"]
)

online_security = st.selectbox(
    "Online Security",
    ["Yes", "No"]
)

online_backup = st.selectbox(
    "Online Backup",
    ["Yes", "No"]
)

device_protection = st.selectbox(
    "Device Protection",
    ["Yes", "No"]
)

tech_support = st.selectbox(
    "Tech Support",
    ["Yes", "No"]
)

streaming_tv = st.selectbox(
    "Streaming TV",
    ["Yes", "No"]
)

streaming_movies = st.selectbox(
    "Streaming Movies",
    ["Yes", "No"]
)

contract = st.selectbox(
    "Contract",
    ["Month-to-month", "One year", "Two year"]
)

paperless_billing = st.selectbox(
    "Paperless Billing",
    ["Yes", "No"]
)

payment_method = st.selectbox(
    "Payment Method",
    [
        "Mailed check",
        "Electronic check",
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    ]
)

monthly_charges = st.number_input(
    "Monthly Charges",
    min_value=0.0,
    value=50.0
)

total_charges = st.number_input(
    "Total Charges",
    min_value=0.0,
    value=500.0
)

# Feature Engineering

total_services = (
    (1 if online_security == "Yes" else 0)
    + (1 if online_backup == "Yes" else 0)
    + (1 if device_protection == "Yes" else 0)
    + (1 if tech_support == "Yes" else 0)
    + (1 if streaming_tv == "Yes" else 0)
    + (1 if streaming_movies == "Yes" else 0)
)

security_services_count = (
    (1 if online_security == "Yes" else 0)
    + (1 if online_backup == "Yes" else 0)
    + (1 if device_protection == "Yes" else 0)
    + (1 if tech_support == "Yes" else 0)
)

streaming_services_count = (
    (1 if streaming_tv == "Yes" else 0)
    + (1 if streaming_movies == "Yes" else 0)
)

if tenure_months <= 12:
    tenure_group = "0-12"
elif tenure_months <= 24:
    tenure_group = "12-24"
elif tenure_months <= 48:
    tenure_group = "24-48"
else:
    tenure_group = "48-72"

avg_revenue_per_month = total_charges / (tenure_months + 1)

if st.button("Predict Churn"):

    input_data = pd.DataFrame({
        'Gender': [gender],
        'Senior Citizen': [senior_citizen],
        'Partner': [partner],
        'Dependents': [dependents],
        'Tenure Months': [tenure_months],
        'Phone Service': [phone_service],
        'Multiple Lines': [multiple_lines],
        'Internet Service': [internet_service],
        'Online Security': [online_security],
        'Online Backup': [online_backup],
        'Device Protection': [device_protection],
        'Tech Support': [tech_support],
        'Streaming TV': [streaming_tv],
        'Streaming Movies': [streaming_movies],
        'Contract': [contract],
        'Paperless Billing': [paperless_billing],
        'Payment Method': [payment_method],
        'Monthly Charges': [monthly_charges],
        'Total Charges': [total_charges],
        'Total Services': [total_services],
        'Security Services Count': [security_services_count],
        'Streaming Services Count': [streaming_services_count],
        'Tenure Group': [tenure_group],
        'Avg Revenue Per Month': [avg_revenue_per_month]
    })

    processed_data = preprocessor.transform(input_data)

    prediction = model.predict(processed_data)[0]
    probability = model.predict_proba(processed_data)[0][1]

    st.subheader("Prediction Result")

    if prediction == 1:
        st.error("Customer is likely to Churn")
    else:
        st.success("Customer is likely to Stay")

    st.write(f"Churn Probability: {probability:.2%}")

Overwriting app.py


In [133]:
with open("app.py", "r") as f:
    print(f.read())

import streamlit as st
import pandas as pd
import joblib

# Load model and preprocessor
model = joblib.load("model.pkl")
preprocessor = joblib.load("preprocessor.pkl")

st.title("Customer Churn Intelligence Platform")

st.write("Predict whether a customer will churn or not.")

st.success("Model loaded successfully!")

st.header("Customer Information")

gender = st.selectbox("Gender", ["Male", "Female"])

senior_citizen = st.selectbox("Senior Citizen", ["No", "Yes"])

partner = st.selectbox("Partner", ["No", "Yes"])

dependents = st.selectbox("Dependents", ["No", "Yes"])

tenure_months = st.number_input(
    "Tenure Months",
    min_value=0,
    max_value=72,
    value=12
)

phone_service = st.selectbox(
    "Phone Service",
    ["Yes", "No"]
)

multiple_lines = st.selectbox(
    "Multiple Lines",
    ["No", "Yes", "No phone service"]
)

internet_service = st.selectbox(
    "Internet Service",
    ["DSL", "Fiber optic", "No"]
)

online_security = st.selectbox(
    "Online Security",
   

In [134]:
with open("app.py", "r") as f:
    print(f.read()[-1000:])

ech Support': [tech_support],
        'Streaming TV': [streaming_tv],
        'Streaming Movies': [streaming_movies],
        'Contract': [contract],
        'Paperless Billing': [paperless_billing],
        'Payment Method': [payment_method],
        'Monthly Charges': [monthly_charges],
        'Total Charges': [total_charges],
        'Total Services': [total_services],
        'Security Services Count': [security_services_count],
        'Streaming Services Count': [streaming_services_count],
        'Tenure Group': [tenure_group],
        'Avg Revenue Per Month': [avg_revenue_per_month]
    })

    processed_data = preprocessor.transform(input_data)

    prediction = model.predict(processed_data)[0]
    probability = model.predict_proba(processed_data)[0][1]

    st.subheader("Prediction Result")

    if prediction == 1:
        st.error("Customer is likely to Churn")
    else:
        st.success("Customer is likely to Stay")

    st.write(f"Churn Probability: {probability:.2%}")


In [135]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
xgboost
joblib

Overwriting requirements.txt
